In [12]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter


SECCIÓN 1: FUNCIONES DE AYUDA PARA ANÁLISIS Y VISUALIZACIÓN

In [13]:


def analizar_y_visualizar_red(G, titulo_red, layout_func=nx.spring_layout, es_bipartita=False, tipo_nodo_col=None):
    """
    Función genérica para analizar métricas clave y generar una visualización de una red.
    """
    print(f"\n--- Análisis de la Red: {titulo_red} ---")
    
    # 1. Información básica de la red
    num_nodos = G.number_of_nodes()
    num_aristas = G.number_of_edges()
    print(f"Número de Nodos: {num_nodos}")
    print(f"Número de Aristas: {num_aristas}")

    if not G.nodes():
        print("La red está vacía. No se puede continuar el análisis.")
        return

    # Si la red no es bipartita, realizamos análisis más profundos
    if not es_bipartita:
        # 2. Componentes Conexas
        num_componentes = nx.number_connected_components(G)
        print(f"Número de Componentes Conexas: {num_componentes}")
        
        # Tomamos la componente gigante para análisis más detallados
        componentes = sorted(nx.connected_components(G), key=len, reverse=True)
        G_gigante = G.subgraph(componentes[0])
        print(f"Tamaño de la Componente Gigante: {G_gigante.number_of_nodes()} nodos")

        # 3. Detección de Comunidades (usando el algoritmo de Louvain)
        print("\nDetectando comunidades...")
        comunidades = nx.community.louvain_communities(G_gigante, weight='weight' if 'weight' in list(G.edges(data=True))[0][-1] else None)
        print(f"Se encontraron {len(comunidades)} comunidades en la componente gigante.")
        
        # Asignar a cada nodo su comunidad para colorear
        colores_nodos = {}
        for i, comunidad in enumerate(comunidades):
            for nodo in comunidad:
                colores_nodos[nodo] = i

    # 4. Visualización
    print("Generando visualización...")
    plt.figure(figsize=(20, 20))
    pos = layout_func(G) # Calcular posiciones de los nodos

    # Configuración de colores y tamaños
    if es_bipartita and tipo_nodo_col:
        colores = ['skyblue' if G.nodes[n][tipo_nodo_col] == 'Autor' else 'lightgreen' for n in G.nodes()]
        nodos_autores = [n for n, d in G.nodes(data=True) if d[tipo_nodo_col] == 'Autor']
        nodos_otros = [n for n, d in G.nodes(data=True) if d[tipo_nodo_col] != 'Autor']
        node_size_map = {n: 50 for n in nodos_autores}
        node_size_map.update({n: 200 for n in nodos_otros})
        sizes = [node_size_map.get(n, 100) for n in G.nodes()]
    elif not es_bipartita:
        colores = [colores_nodos.get(n, -1) for n in G.nodes()] # Usar colores de comunidad
        grados = dict(G.degree())
        sizes = [grados.get(n, 1) * 50 for n in G.nodes()] # Tamaño por grado
    else:
        colores = 'skyblue'
        sizes = 100

    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colores, cmap=plt.cm.jet, alpha=0.8)
    
    # Dibujar aristas con pesos (si existen)
    if 'weight' in list(G.edges(data=True))[0][-1]:
        pesos = [d['weight'] for (u, v, d) in G.edges(data=True)]
        nx.draw_networkx_edges(G, pos, width=[w * 0.5 for w in pesos], alpha=0.3, edge_color='gray')
    else:
        nx.draw_networkx_edges(G, pos, alpha=0.3, edge_color='gray')

    # Etiquetas para nodos más grandes (para no saturar el gráfico)
    nodos_grandes = [n for n, size in zip(G.nodes(), sizes) if size > sorted(sizes)[-20:].pop(0)]
    etiquetas = {n: n for n in nodos_grandes}
    nx.draw_networkx_labels(G, pos, labels=etiquetas, font_size=10, font_color='black')
    
    plt.title(titulo_red, size=20)
    plt.axis('off')
    plt.savefig(f"{titulo_red.replace(' ', '_').lower()}.png", bbox_inches='tight')
    print(f"Gráfico guardado como '{titulo_red.replace(' ', '_').lower()}.png'")
    plt.close()

SECCIÓN 2: CARGA Y ANÁLISIS DE CADA RED

In [14]:


# --- 1. Análisis de Red de Coautoría ---
try:
    df_nodos_coautoria = pd.read_csv('redes/1_red_coautoria_nodos.csv')
    df_aristas_coautoria = pd.read_csv('redes/1_red_coautoria_aristas.csv')
    
    G_coautoria = nx.from_pandas_edgelist(df_aristas_coautoria, 'Source', 'Target', edge_attr='Weight')
    analizar_y_visualizar_red(G_coautoria, "Red de Coautoria")
    
    # Análisis de centralidad adicional
    print("\nTop 10 Autores por Centralidad de Grado:")
    grado = sorted(G_coautoria.degree(weight='Weight'), key=lambda item: item[1], reverse=True)
    print(pd.DataFrame(grado[:10], columns=['Autor', 'Grado Ponderado']))
    
    print("\nTop 10 Autores por Centralidad de Intermediación:")
    intermediacion = nx.betweenness_centrality(G_coautoria, weight='Weight', normalized=True)
    intermediacion_sorted = sorted(intermediacion.items(), key=lambda item: item[1], reverse=True)
    print(pd.DataFrame(intermediacion_sorted[:10], columns=['Autor', 'Intermediación']))
    
except FileNotFoundError:
    print("Archivos para la Red de Coautoría no encontrados. Saltando análisis.")




--- Análisis de la Red: Red de Coautoria ---
Número de Nodos: 443
Número de Aristas: 765
Número de Componentes Conexas: 79
Tamaño de la Componente Gigante: 73 nodos

Detectando comunidades...
Se encontraron 8 comunidades en la componente gigante.
Generando visualización...
Gráfico guardado como 'red_de_coautoria.png'

Top 10 Autores por Centralidad de Grado:
                              Autor  Grado Ponderado
0           Julián Bravo Castillero               26
1  Reinaldo Rodr(cid:237)guez Ramos               26
2               Raúl Guinovart Díaz               20
3           Sandy Sánchez Domínguez               18
4            Gladys Linares Fleites               16
5            Mijail Borges Quintana               16
6  Carlos Rafael Sebrango Rodríguez               15
7       Miguel Angel Borges Trenard               15
8           Yanet Rodríguez Sarabia               14
9         Antonio Iván Ruiz Chaveco               14

Top 10 Autores por Centralidad de Intermediación:
    


--- Análisis de la Red: Red de Afiliacion Autor-Institucion ---
Número de Nodos: 808
Número de Aristas: 1372
Generando visualización...
Gráfico guardado como 'red_de_afiliacion_autor-institucion.png'


c:\Users\crist\AppData\Local\Programs\Python\Python310\lib\site-packages\networkx\drawing\nx_pylab.py:457: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  node_collection = ax.scatter(


In [16]:
# --- 3. Análisis de Red de Palabras Clave ---
try:
    df_nodos_kw = pd.read_csv('redes/3_red_keywords_nodos.csv')
    df_aristas_kw = pd.read_csv('redes/3_red_keywords_aristas.csv')
    
    G_kw = nx.from_pandas_edgelist(df_aristas_kw, 'Source', 'Target', edge_attr='Weight')
    analizar_y_visualizar_red(G_kw, "Red de Co-ocurrencia de Palabras Clave")
    
    print("\nTop 10 Palabras Clave por Centralidad de Grado:")
    grado_kw = sorted(G_kw.degree(weight='Weight'), key=lambda item: item[1], reverse=True)
    print(pd.DataFrame(grado_kw[:10], columns=['Palabra Clave', 'Grado Ponderado']))
    
except FileNotFoundError:
    print("Archivos para la Red de Palabras Clave no encontrados. Saltando análisis.")


--- Análisis de la Red: Red de Co-ocurrencia de Palabras Clave ---
Número de Nodos: 597
Número de Aristas: 996
Número de Componentes Conexas: 129
Tamaño de la Componente Gigante: 43 nodos

Detectando comunidades...
Se encontraron 5 comunidades en la componente gigante.
Generando visualización...
Gráfico guardado como 'red_de_co-ocurrencia_de_palabras_clave.png'

Top 10 Palabras Clave por Centralidad de Grado:
            Palabra Clave  Grado Ponderado
0                covid-19               23
1          criptoanálisis               16
2               seguridad               12
3            competencias               12
4      algoritmo genético               11
5                 covid19               10
6                 android               10
7  certificados digitales               10
8                 cifrado               10
9            codificación               10


In [17]:
# --- 4. Análisis de Red Temática (Bipartita) ---
try:
    df_nodos_tematica = pd.read_csv('redes/4_red_tematica_nodos.csv')
    df_aristas_tematica = pd.read_csv('redes/4_red_tematica_aristas.csv')
    
    G_tematica = nx.from_pandas_edgelist(df_aristas_tematica, 'Source', 'Target')
    nx.set_node_attributes(G_tematica, pd.Series(df_nodos_tematica.TipoNodo, index=df_nodos_tematica.Id).to_dict(), 'TipoNodo')

    analizar_y_visualizar_red(G_tematica, "Red de Afiliacion Autor-Tematica", es_bipartita=True, tipo_nodo_col='TipoNodo', layout_func=nx.kamada_kawai_layout)

except FileNotFoundError:
    print("Archivos para la Red Temática no encontrados. Saltando análisis.")



--- Análisis de la Red: Red de Afiliacion Autor-Tematica ---
Número de Nodos: 680
Número de Aristas: 645
Generando visualización...
Gráfico guardado como 'red_de_afiliacion_autor-tematica.png'


c:\Users\crist\AppData\Local\Programs\Python\Python310\lib\site-packages\networkx\drawing\nx_pylab.py:457: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  node_collection = ax.scatter(


 SECCIÓN 3: PROYECCIONES DE LA RED BIPARTITA DE AFILIACIÓN 

In [25]:
# --- [ANÁLISIS 2 y 5 COMBINADOS] Red de Afiliación y sus Proyecciones ---
print("\n--- [ANÁLISIS 2/4] Red de Afiliación y Proyecciones ---")
try:
    # Carga los archivos de la red de afiliación
    df_nodos_afiliacion = pd.read_csv('redes/2_red_afiliacion_nodos.csv')
    df_aristas_afiliacion = pd.read_csv('redes/2_red_afiliacion_aristas.csv')

    # --- MÉTODO ROBUSTO DE CONSTRUCCIÓN DE GRAFO ---
    G_afiliacion = nx.Graph()
    for index, row in df_nodos_afiliacion.iterrows():
        G_afiliacion.add_node(row['Id'], label=row['Label'], TipoNodo=row['TipoNodo'])
    for index, row in df_aristas_afiliacion.iterrows():
        G_afiliacion.add_edge(row['Source'], row['Target'])
    print("Grafo de Afiliación construido correctamente.")
    print(f"Total de Nodos: {G_afiliacion.number_of_nodes()}, Total de Aristas: {G_afiliacion.number_of_edges()}")

    # --- Visualización de la Red Bipartita Original ---
    analizar_y_visualizar_red(G_afiliacion, "Red de Afiliacion Autor-Institucion", es_bipartita=True, tipo_nodo_col='TipoNodo', layout_func=nx.kamada_kawai_layout)

    # --- ANÁLISIS DE PROYECCIONES (VERSIÓN CORREGIDA Y ROBUSTA) ---
    print("\n--- Proyecciones de la Red de Afiliación ---")
    nodos_autores = {n for n, d in G_afiliacion.nodes(data=True) if d.get('TipoNodo') == 'Autor'}
    nodos_instituciones = {n for n, d in G_afiliacion.nodes(data=True) if d.get('TipoNodo') == 'Afiliacion'}
    print(f"Nodos de tipo 'Autor' encontrados: {len(nodos_autores)}")
    print(f"Nodos de tipo 'Afiliacion' encontrados: {len(nodos_instituciones)}")

    # --- 2. Proyección sobre Autores ---
    if len(nodos_autores) > 0:
        print("\n[Proyección 1] Creando Red de Autores por Afiliación Compartida...")
        G_proy_autores = nx.bipartite.weighted_projected_graph(G_afiliacion, nodos_autores)
        print(f"Red de Similitud Institucional creada con {G_proy_autores.number_of_nodes()} autores y {G_proy_autores.number_of_edges()} colaboraciones indirectas.")
        if G_proy_autores.number_of_edges() > 0:
            analizar_y_visualizar_red(G_proy_autores, "Proyeccion - Autores por Institucion Compartida")
        else:
            print("La red de proyección de autores no tiene aristas.")

    # --- 3. Proyección sobre Instituciones ---
    if len(nodos_instituciones) > 0:
        print("\n[Proyección 2] Creando Red de Instituciones por Autor Compartido...")
        G_proy_instituciones = nx.bipartite.weighted_projected_graph(G_afiliacion, nodos_instituciones)
        print(f"Red de Colaboración Institucional creada con {G_proy_instituciones.number_of_nodes()} instituciones y {G_proy_instituciones.number_of_edges()} vínculos.")
        if G_proy_instituciones.number_of_edges() > 0:
            print("Generando visualización de la red de instituciones...")
            num_nodos_inst = G_proy_instituciones.number_of_nodes()
            fig_size = max(12, num_nodos_inst * 0.8)
            plt.figure(figsize=(fig_size, fig_size))
            pos = nx.spring_layout(G_proy_instituciones, k=0.8, iterations=50)
            grados = dict(G_proy_instituciones.degree(weight='weight'))
            node_sizes = [grados.get(n, 1) * 100 for n in G_proy_instituciones.nodes()]
            font_sizes = max(8, 20 - num_nodos_inst * 0.2)
            nx.draw_networkx_nodes(G_proy_instituciones, pos, node_size=node_sizes, node_color='lightgreen', alpha=0.9)
            pesos = [d['weight'] for u, v, d in G_proy_instituciones.edges(data=True)]
            nx.draw_networkx_edges(G_proy_instituciones, pos, width=[w * 0.5 for w in pesos], alpha=0.4, edge_color='gray')
            nx.draw_networkx_labels(G_proy_instituciones, pos, font_size=font_sizes, font_color='black')
            plt.title("Proyeccion - Red de Colaboracion Institucional", size=20)
            plt.margins(0.1, 0.1)
            plt.savefig("proyeccion_red_instituciones.png", bbox_inches='tight', dpi=150)
            print("Gráfico 'proyeccion_red_instituciones.png' guardado.")
            plt.close()
        else:
            print("La red de proyección de instituciones no tiene aristas.")

except FileNotFoundError:
    print("Archivos para la Red de Afiliación no encontrados. Saltando análisis.")


--- [ANÁLISIS 2/4] Red de Afiliación y Proyecciones ---
Grafo de Afiliación construido correctamente.
Total de Nodos: 808, Total de Aristas: 1372

--- Análisis de la Red: Red de Afiliacion Autor-Institucion ---
Número de Nodos: 808
Número de Aristas: 1372
Generando visualización...


c:\Users\crist\AppData\Local\Programs\Python\Python310\lib\site-packages\networkx\drawing\nx_pylab.py:457: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  node_collection = ax.scatter(


Gráfico guardado como 'red_de_afiliacion_autor-institucion.png'

--- Proyecciones de la Red de Afiliación ---
Nodos de tipo 'Autor' encontrados: 456
Nodos de tipo 'Afiliacion' encontrados: 352

[Proyección 1] Creando Red de Autores por Afiliación Compartida...
Red de Similitud Institucional creada con 456 autores y 1672 colaboraciones indirectas.

--- Análisis de la Red: Proyeccion - Autores por Institucion Compartida ---
Número de Nodos: 456
Número de Aristas: 1672
Número de Componentes Conexas: 63
Tamaño de la Componente Gigante: 234 nodos

Detectando comunidades...
Se encontraron 12 comunidades en la componente gigante.
Generando visualización...
Gráfico guardado como 'proyeccion_-_autores_por_institucion_compartida.png'

[Proyección 2] Creando Red de Instituciones por Autor Compartido...
Red de Colaboración Institucional creada con 352 instituciones y 1464 vínculos.
Generando visualización de la red de instituciones...
Gráfico 'proyeccion_red_instituciones.png' guardado.
